In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel, VarianceThreshold, SelectPercentile
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from dotenv import load_dotenv
import joblib
from gene_thesaurus import GeneThesaurus


In [2]:
load_dotenv()

X_train_path = os.getenv("X_TRAIN_CSV_PATH")
y_train_path = os.getenv("Y_TRAIN_TSV_PATH")
X_test_path = os.getenv("X_TEST_CSV_PATH")
y_test_path = os.getenv("Y_TEST_TSV_PATH")
output_dir = os.getenv("OUTPUT_DIR")
random_seed = int(os.getenv("RANDOM_SEED"))
n_jobs = int(os.getenv("N_JOBS", -1))

# Print configuration
print(f"X_train_path: {X_train_path}")
print(f"y_train_path: {y_train_path}")
print(f"X_test_path: {X_test_path}")
print(f"y_test_path: {y_test_path}")
print(f"output_dir: {output_dir}")
print(f"random_seed: {random_seed}")
print(f"n_jobs: {n_jobs}")

# Hardcoded variables
VARIANCE_THRESHOLD=0.01  # REMOVE NEAR-CONSTANT GENES
FEATURE_SELECTION_PERCENTILE=20  # KEEP TOP n% OF FEATURES

X_train_path: /home/mariya/Data/pallas/nf-core-gexml-output-train/transpose/transposed.csv
y_train_path: /home/mariya/Data/pallas/nf-core-gexml-output-train/class/subtypes.tsv
X_test_path: /home/mariya/Data/pallas/nf-core-gexml-output-test/transpose/transposed.csv
y_test_path: /home/mariya/Data/pallas/nf-core-gexml-output-test/class/subtypes.tsv
output_dir: /home/mariya/Data/pallas/training
random_seed: 42
n_jobs: 10


In [3]:
# Read in X_train and y_train
X_train = pd.read_csv(X_train_path, index_col=0)
X_train


,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
id,,,,,,,,,,,,,,,,,,,,,
Case_1,-1.270355,-1.270355,3.501935,3.838579,3.202853,5.353419,-1.071296,3.009842,4.329323,4.921633,...,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355
Case_10,-0.104178,-1.270355,4.401356,3.716347,4.195322,2.971083,1.730790,3.187137,4.000332,6.006571,...,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355
Case_102,-0.797662,-1.270355,3.945437,3.677019,3.684148,3.273750,0.500723,3.990458,3.364442,6.732017,...,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355
Case_104,-1.270355,-1.270355,4.822167,3.658020,4.234307,5.872168,0.698601,4.068325,6.592908,5.138176,...,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355
Case_106,-0.115145,-1.270355,4.774554,2.201566,2.954007,1.485179,-1.029297,0.713269,3.420874,6.820978,...,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SJTALL033944_D1,3.679231,3.555146,2.620108,3.873038,3.823903,3.583644,1.410226,3.893350,3.547426,4.609713,...,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355
SJTALL034114_D1,-0.266784,-1.270355,4.092312,4.218846,4.332038,4.654840,3.563651,4.438379,4.136140,6.471575,...,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355
SJTALL034427_D1,0.762654,-0.575246,4.316558,4.326894,4.377870,8.415231,0.764747,5.261284,6.114764,6.121250,...,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355


In [4]:
y_train = pd.read_csv(y_train_path, sep="\t", index_col=0)
y_train

,class
sample,
SJAML031398_D1,"Disease,AML,KMT2A-r"
SJBALL032059_D1,"Disease,B-ALL,PAX5alt"
SJAML074187_D1,"Disease,AML,KMT2A-r"
SJBALL031613_D1,"Disease,B-ALL,PAX5 P80R"
SJBALL021023_D1,"Disease,B-ALL,aneuploidy,high hyperdiploid"
...,...
SJHBBBN9RBJ,"Disease,B-ALL,aneuploidy,high hyperdiploid"
SJHBBBM5GW9,"Disease,B-ALL,aneuploidy,high hyperdiploid"
SJHBBBM4X52,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"


In [5]:
# align the indices
X_train = X_train.loc[y_train.index]
X_train

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
sample,,,,,,,,,,,,,,,,,,,,,
SJAML031398_D1,-1.068683,-1.270355,3.117074,2.782854,2.522931,7.878627,-0.458110,6.312663,3.909614,5.267971,...,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355
SJBALL032059_D1,-0.636940,-0.761472,5.082960,2.852008,2.565583,3.849803,2.268138,2.211968,4.494831,6.957567,...,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355
SJAML074187_D1,-0.603813,-1.270355,5.253653,4.286418,3.131662,5.415364,0.039690,6.522635,5.115177,5.832558,...,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355
SJBALL031613_D1,-0.360986,0.360234,4.147329,2.805418,3.095724,4.953006,3.215205,3.424850,4.314127,5.851993,...,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355
SJBALL021023_D1,0.114667,-1.270355,4.516719,2.521878,2.611377,6.210731,1.607525,3.881327,4.946072,5.363263,...,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SJHBBBN9RBJ,2.027061,-1.270355,4.358609,2.750648,4.378296,1.184531,3.022314,2.319762,4.915188,6.083464,...,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355
SJHBBBM5GW9,-0.865513,-1.270355,3.923224,4.484584,3.620145,7.208626,2.164281,3.362869,2.466058,6.236636,...,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355
SJHBBBM4X52,-0.979045,-1.270355,3.817093,4.369270,3.891738,7.280500,0.877575,3.867617,4.986625,6.064771,...,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355,-1.270355


In [6]:
X_test = pd.read_csv(X_test_path, index_col=0)
y_test = pd.read_csv(y_test_path, sep="\t", index_col=0)
y_test

,class
sample,
SJBALL020803,"Disease,B-ALL,aneuploidy,high hyperdiploid"
SJBALL014221,"Disease,B-ALL,BCR::ABL1-group,BCR::ABL1"
SJBALL021234,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"
SJHYPER108,"Disease,B-ALL,aneuploidy,high hyperdiploid"
SJBALL021118,"Disease,B-ALL,BCR::ABL1-group,BCR::ABL1-like"
...,...
GE3320,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"
ALLT-358,"Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1"
ALLT-343,"Disease,B-ALL,BCR::ABL1-group,BCR::ABL1"


In [7]:
# align the indices
X_test = X_test.loc[y_test.index]
X_test

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
sample,,,,,,,,,,,,,,,,,,,,,
SJBALL020803,1.904466,-1.245073,4.671270,3.642147,4.074731,6.084868,1.299425,2.825236,4.009486,6.708023,...,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073
SJBALL014221,-0.994755,-1.245073,4.159572,4.664387,4.305278,8.595558,6.788747,6.205900,4.710307,6.261220,...,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073
SJBALL021234,-1.145694,-1.245073,4.634658,3.512398,3.579004,2.846397,0.559841,1.832314,4.544552,5.189278,...,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073
SJHYPER108,0.184327,-1.245073,4.028165,4.399836,4.750806,4.097145,-1.166784,1.949442,4.777032,6.432371,...,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073
SJBALL021118,-0.541454,-1.245073,4.615323,3.676348,4.089415,4.216980,2.487864,2.311113,4.386339,7.065609,...,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GE3320,0.382887,-1.245073,4.662200,1.892010,2.563201,2.831249,-1.245073,2.133063,2.999071,7.616085,...,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073
ALLT-358,-0.258104,-1.245073,5.217416,4.542182,4.916120,6.180057,2.100939,2.875471,4.777466,6.246564,...,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073
ALLT-343,0.011150,-0.806598,4.078723,3.838250,4.229574,2.812869,0.605629,2.562657,3.747731,6.682622,...,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073,-1.245073


In [8]:
# Get least frequently ocurring class count
min_class_count = y_train.value_counts().min()
min_class_count

np.int64(3)

In [9]:
y_train = y_train.iloc[:, 0].values  # Convert to 1D numpy array
y_test = y_test.iloc[:, 0].values
y_test

array(['Disease,B-ALL,aneuploidy,high hyperdiploid',
       'Disease,B-ALL,BCR::ABL1-group,BCR::ABL1',
       'Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1',
       'Disease,B-ALL,aneuploidy,high hyperdiploid',
       'Disease,B-ALL,BCR::ABL1-group,BCR::ABL1-like',
       'Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1',
       'Disease,B-ALL,aneuploidy,high hyperdiploid',
       'Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1',
       'Disease,B-ALL,BCR::ABL1-group,BCR::ABL1-like',
       'Disease,B-ALL,BCR::ABL1-group,BCR::ABL1-like',
       'Disease,B-ALL,aneuploidy,high hyperdiploid',
       'Disease,B-ALL,PAX5alt',
       'Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1',
       'Disease,B-ALL,BCR::ABL1-group,BCR::ABL1',
       'Disease,B-ALL,BCR::ABL1-group,BCR::ABL1', 'Disease,B-ALL,PAX5alt',
       'Disease,B-ALL,DUX4-r',
       'Disease,B-ALL,ETV6::RUNX1-group,ETV6::RUNX1-like',
       'Disease,B-ALL,TCF3::PBX1',
       'Disease,B-ALL,BCR::ABL1-group,BCR::ABL1-like',
       'Disease,B-AL

In [10]:
N_ITER=30
N_REPEATS=1 # WE WILL VALIDATE FINAL PARAMS USING MORE REPEATS

# ----------------------
# Pipeline: Variance filter -> RF feature selection -> RF classifier
# ----------------------
pipeline = Pipeline([
    ("variance_filter", VarianceThreshold(threshold=VARIANCE_THRESHOLD)), # remove near-constant genes
    ("feature_selection", SelectPercentile(percentile=FEATURE_SELECTION_PERCENTILE)),  # keep only top 10% of features
    ("rf", RandomForestClassifier(random_seed, n_jobs=-n_jobs))
])

# ----------------------
# Hyperparameter search space
# ----------------------
param_grid = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [None, 20, 50],
    'rf__max_features': ['sqrt', 'log2'],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 5],
    'rf__class_weight': [None, 'balanced']
}

# ----------------------
# Cross-validation + randomized search
# ----------------------
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=N_ITER,
    cv=cv,
    scoring="f1_weighted",   # good for multiclass imbalance
    n_jobs=-n_jobs,
    random_state=random_seed,
    verbose=1
)

search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print("Best CV (F1-weighted):", search.best_score_)
best_pipeline = search.best_estimator_

## SAVE EVERYTHING
# Save best hyperparameters
joblib.dump(search.best_params_, os.path.join(output_dir, "best_hyperparameters.pkl"))

# Save features - USE BEST_PIPELINE INSTEAD OF PIPELINE
# First get features that passed the variance threshold
variance_mask = best_pipeline.named_steps['variance_filter'].get_support()
remaining_features = X_train.columns[variance_mask]

# Then get features selected by SelectFromModel
selected_mask = best_pipeline.named_steps['feature_selection'].get_support()
selected_features = remaining_features[selected_mask]

joblib.dump(selected_features, os.path.join(output_dir, "selected_features.pkl"))

# Save the whole pipeline
joblib.dump(best_pipeline, os.path.join(output_dir, "best_pipeline.pkl"))

Fitting 3 folds for each of 30 candidates, totalling 90 fits
Best hyperparameters: {'rf__n_estimators': 200, 'rf__min_samples_split': 10, 'rf__min_samples_leaf': 5, 'rf__max_features': 'sqrt', 'rf__max_depth': 50, 'rf__class_weight': 'balanced'}
Best CV (F1-weighted): 0.9103910485924246


['/home/mariya/Data/pallas/training/best_pipeline.pkl']

In [11]:
# Cross-validate again with more repeats and larger estimators
N_REPEATS=10

# Use just the selected features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# Read in "best_hyperparameters.pkl"
with open(os.path.join(output_dir, "best_hyperparameters.pkl"), "rb") as f:
    best_params = joblib.load(f)

best_params = {k.replace('rf__', ''): v for k, v in best_params.items()}
best_params['n_estimators'] = 500  # Use more trees for final model
print("Final model parameters:", best_params)
final_rf = RandomForestClassifier(**best_params,
                            n_jobs=n_jobs,
                            random_state=random_seed)

# Cross-validate again with more repeats
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

cv_scores = cross_val_score(final_rf,
                           X_train_selected,
                           y_train,
                           cv=cv,
                           scoring="f1_weighted",
                           n_jobs=-n_jobs,
                           verbose=1)

print(f"Cross-validation F1-weighted: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Train the final model on all training data
final_rf.fit(X_train_selected, y_train)

# Save the final model
joblib.dump(final_rf, os.path.join(output_dir, "final_model.pkl"))

Final model parameters: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 50, 'class_weight': 'balanced'}


[Parallel(n_jobs=-10)]: Using backend LokyBackend with 5 concurrent workers.
[Parallel(n_jobs=-10)]: Done  30 out of  30 | elapsed:  3.9min finished


Cross-validation F1-weighted: 0.916 ± 0.009


['/home/mariya/Data/pallas/training/final_model.pkl']

In [ ]:
# ----------------------
# Evaluate final model on test set
# ----------------------

# Use selected features for test set
y_pred_final = final_rf.predict(X_test_selected)

# Print size of selected features
print(f"Number of selected features: {len(selected_features)}")

# Print shape of X_test_selected
print(f"Shape of X_test_selected: {X_test_selected.shape}")

print("\nTest set classification report (final optimized model):")
print(classification_report(y_test, y_pred_final, digits=3))


Number of selected features: 3931
Shape of X_test_selected: (513, 3931)

Test set classification report (final optimized model):
                                                  precision    recall  f1-score   support

                       Disease,AML,APL-PML::RARA      1.000     0.500     0.667         2
                      Disease,AML,CBFA2T3::GLIS2      0.667     1.000     0.800         2
                         Disease,AML,CBFB::MYH11      1.000     1.000     1.000         9
                            Disease,AML,CEBPAalt      1.000     0.800     0.889         5
                             Disease,AML,KMT2A-r      0.867     0.812     0.839        16
                             Disease,AML,NPM1alt      1.000     0.625     0.769         8
                             Disease,AML,NUP98-r      1.000     0.714     0.833         7
                      Disease,AML,RUNX1::RUNX1T1      0.933     0.933     0.933        15
         Disease,B-ALL,BCR::ABL1-group,BCR::ABL1      0.750 

In [14]:

# ----------------------
# Extract top genes from final model
# ----------------------
# Get feature importances from final RF
importances = final_rf.feature_importances_

# Rank genes by importance
gene_importances = pd.DataFrame({
    "Gene": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

gt = GeneThesaurus()
translated = gt.translate_genes(gene_importances["Gene"].tolist(),
                               source="ensembl_id", target="symbol")
gene_importances["Gene"] = gene_importances["Gene"].map(translated)

print("\nTop 20 genes driving classification (final model):")
print(gene_importances.head(20))


Top 20 genes driving classification (final model):
         Gene  Importance
3268    NUTM1    0.005771
1889   MEGF10    0.005068
3830    APELA    0.003940
2203    AGAP1    0.003675
1830    MEIS1    0.003586
353   RUNX1T1    0.002926
706     KCNN1    0.002632
3315     PBX1    0.002585
3285   ANKS1B    0.002562
546     SALL4    0.002548
2161   TSPAN7    0.002514
2998  KBTBD11    0.002426
1036     CNN3    0.002212
3252   KCNJ12    0.002165
1413   ATP8A2    0.002147
2192     DGKI    0.002107
1436    MYH11    0.002062
348     HOXA9    0.002047
2247   DONSON    0.001997
1197    ATXN1    0.001973


In [15]:
# How many of the feature importances are zero?
num_zero_importances = np.sum(importances == 0)
print(f"\nNumber of features with zero importance: {num_zero_importances} out of {len(importances)}")

# Print lowest importance genes as well
print("\nBottom 20 genes by importance (final model):")
print(gene_importances.tail(20))


Number of features with zero importance: 4 out of 3931

Bottom 20 genes by importance (final model):
          Gene    Importance
2929     BRMS1  4.188505e-06
3807       MIF  4.140330e-06
70       APBA3  3.877452e-06
1373     BARX1  3.733394e-06
3340     FFAR4  3.618487e-06
3928      F8A1  3.532297e-06
460    LRRFIP2  2.825473e-06
1815   CREB3L4  2.743391e-06
3169    OR52K2  2.728419e-06
1932     FATE1  2.426159e-06
1172      CHD6  2.377005e-06
2846    DPAGT1  2.316390e-06
1774    SLC6A3  1.805014e-06
3471  KRTAP5-3  1.109546e-06
3907     ORAI1  1.773943e-20
3120     SOCS4  8.865781e-21
3665    XAGE1A  0.000000e+00
3366     QRFPR  0.000000e+00
3388      SBK2  0.000000e+00
937        IL4  0.000000e+00
